In [ ]:
%pip -q install easyocr opencv-python-headless pillow matplotlib pandas deepface tf-keras

In [ ]:
import os
import re
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

%matplotlib inline
print("Libraries imported successfully.")

In [ ]:
def load_image(image_path):
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

def show_image(image, title="Image", figsize=(9, 6)):
    plt.figure(figsize=figsize)
    plt.imshow(image)
    plt.title(title)
    plt.axis("off")
    plt.show()

# Optional Google Colab upload:
# from google.colab import files
# uploaded = files.upload()
# IMAGE_PATH = next(iter(uploaded))

In [ ]:
import easyocr

# English OCR model; set gpu=True only if a compatible GPU is available.
ocr_reader = easyocr.Reader(['en'], gpu=False)
print("EasyOCR reader is ready.")

In [ ]:
def preprocess_for_ocr(rgb_image):
    """Upscale and enhance contrast to help OCR on small printed labels."""
    gray = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2GRAY)
    height, width = gray.shape
    scale = 2 if max(height, width) < 1800 else 1
    if scale > 1:
        gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    # Light denoising and adaptive thresholding can help with uneven lighting.
    denoised = cv2.bilateralFilter(gray, 9, 75, 75)
    enhanced = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(denoised)
    thresholded = cv2.adaptiveThreshold(
        enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 31, 11
    )
    return thresholded

def read_medicine_label(image_path, show_preprocessing=True):
    image = load_image(image_path)
    processed = preprocess_for_ocr(image)

    if show_preprocessing:
        show_image(image, "Original medicine label")
        show_image(processed, "Preprocessed image")

    results = ocr_reader.readtext(processed, detail=1, paragraph=False)
    rows = []
    for box, text, confidence in results:
        rows.append({
            "Recognized text": text,
            "OCR confidence": round(float(confidence), 3)
        })

    result_df = pd.DataFrame(rows)
    extracted_text = "\n".join(row["Recognized text"] for row in rows)
    return extracted_text, result_df

# Provide the path to a medicine-bottle image:
MEDICINE_IMAGE_PATH = "medicine_label.jpg"  # Change this to your image filename

# Uncomment after setting the correct path:
# medicine_text, medicine_results = read_medicine_label(MEDICINE_IMAGE_PATH)
# display(medicine_results)
# print("\nFull OCR text:\n", medicine_text)

In [ ]:
def suggest_label_fields(extracted_text):
    """Find likely label fields using simple patterns; verify every result manually."""
    lines = [line.strip() for line in extracted_text.splitlines() if line.strip()]
    text = "\n".join(lines)

    patterns = {
        "Expiry date": r"(?i)\b(?:exp(?:iry)?\.?|use before)\s*[:.-]?\s*([A-Z0-9/.-]{3,})",
        "Manufacturing date": r"(?i)\b(?:mfg|manufactured)\.?\s*[:.-]?\s*([A-Z0-9/.-]{3,})",
        "Batch number": r"(?i)\b(?:batch|b\.\s*no\.?|lot)\s*[:.-]?\s*([A-Z0-9-]{2,})",
        "Strength": r"(?i)\b(\d+(?:\.\d+)?\s?(?:mg|mcg|g|ml|%))\b",
    }

    suggestions = {}
    for field, pattern in patterns.items():
        match = re.search(pattern, text)
        suggestions[field] = match.group(1) if match else "Not detected"

    # The first non-empty OCR line is only a possible product-name candidate.
    suggestions["Possible medicine name (first OCR line)"] = lines[0] if lines else "Not detected"
    return pd.DataFrame(
        [{"Field": key, "OCR suggestion": value} for key, value in suggestions.items()]
    )

# Example usage after running OCR:
# display(suggest_label_fields(medicine_text))

In [ ]:
sample_label_text = """PARACETAMOL TABLETS
500 mg
Batch No: AB1234
Mfg: 01/2026
Exp: 12/2027
"""
display(suggest_label_fields(sample_label_text))

In [ ]:
from deepface import DeepFace
print("DeepFace imported successfully.")

In [ ]:
def analyze_facial_expression(image_path, detector_backend="opencv"):
    """Analyze the most prominent face and return DeepFace's expression scores."""
    image = load_image(image_path)
    show_image(image, "Input face image")

    analysis = DeepFace.analyze(
        img_path=image_path,
        actions=["emotion"],
        detector_backend=detector_backend,
        enforce_detection=True,
        silent=True
    )

    # DeepFace may return a list when more than one face is detected.
    face_result = analysis[0] if isinstance(analysis, list) else analysis
    emotion_scores = face_result.get("emotion", {})
    dominant = face_result.get("dominant_emotion", "unknown")

    scores_df = pd.DataFrame(
        [{"Expression": name, "Score (%)": round(float(score), 2)}
         for name, score in emotion_scores.items()]
    ).sort_values("Score (%)", ascending=False)

    return dominant, scores_df, face_result

# Provide the path to a face image:
FACE_IMAGE_PATH = "face.jpg"  # Change this to your image filename

# Uncomment after setting the correct path:
# dominant_emotion, emotion_scores, face_details = analyze_facial_expression(FACE_IMAGE_PATH)
# print("Most likely expression category:", dominant_emotion)
# display(emotion_scores)

In [ ]:
def supportive_assistant_response(expression):
    """Generate a gentle response without claiming to know the user's feelings."""
    responses = {
        "happy": "You appear to have a positive expression. Would you like to share what's going well?",
        "sad": "You may be looking a little down. Would you like to talk about what's on your mind?",
        "angry": "You may be showing signs of frustration. Would it help to pause or talk through what's bothering you?",
        "fear": "You may look concerned. Is there anything you'd like help thinking through?",
        "surprise": "You seem surprised. Would you like to tell me what happened?",
        "disgust": "Your expression may suggest discomfort. Is something bothering you?",
        "neutral": "Your expression looks fairly neutral. How are you feeling right now?"
    }
    return responses.get(
        str(expression).lower(),
        "I couldn't confidently interpret the expression. How are you feeling?"
    )

# Example response:
for category in ["happy", "sad", "angry", "neutral"]:
    print(f"{category.title()}: {supportive_assistant_response(category)}")

In [ ]:
# Complete emotion-aware assistant run:
# dominant_emotion, emotion_scores, face_details = analyze_facial_expression(FACE_IMAGE_PATH)
# print("\nEstimated expression:", dominant_emotion)
# display(emotion_scores)
# print("\nAssistant:", supportive_assistant_response(dominant_emotion))